# AgriVision — Eksperimen Pra-pemrosesan Citra pada MobileNetV2 (v3)
### Ablasi enhancement untuk klasifikasi penyakit daun tomat: generalisasi lab → lapangan

**Pertanyaan penelitian:** apakah pra-pemrosesan citra (CLAHE, normalisasi warna, sharpening, penekanan latar) mengubah akurasi MobileNetV2 pada foto lab **dan** pada foto lapangan non-lab?

**Catatan kebaruan (jujur):** CLAHE + MobileNetV2 pada tomat sudah dipublikasikan (mis. Ahmed dkk., IEEE Access 2022, hanya pada data lab PlantVillage, selisih ±0,4 poin). Yang diuji di sini adalah **efeknya pada generalisasi lab → lapangan** dengan split bebas kebocoran (per daun) dan set uji lapangan terpisah (PlantDoc) — sesuatu yang tidak ditemukan diuji secara terpisah untuk normalisasi warna dan penekanan latar.

**Desain:**
- Model: MobileNetV2 (ImageNet), 10 kelas tomat (1 sehat + 9 penyakit), masukan 224×224, dua tahap (kepala klasifikasi lalu fine-tuning lapisan atas).
- Data: sama dengan v2 — `lab_*` dari PlantVillage (split per daun), `field_*` dari PlantDoc. Pembagian dan jumlah per kelas dicetak di Step 5.
- Varian pra-pemrosesan (modul `ml-service/app/ml/enhance.py`, dipakai identik di training dan API): `none`, `clahe`, `sog`, `unsharp`, `bg`, `sog_clahe`, `bg_sog_clahe`. Urutan: resize 224 (PIL bilinear) → enhancement → disimpan PNG → dilatih. Augmentasi dan semua hiperparameter **sama** untuk semua varian.
- Pemilihan varian memakai akurasi `field_val`; `lab_test` dan `field_test` hanya dilaporkan. Beberapa seed untuk varian utama; uji McNemar dan bootstrap terhadap baseline.

Jalankan di Colab T4 (Runtime → Run all). Tanpa API key. Perkiraan total 2–3 jam.

In [ ]:
# Cek GPU aktif — kalau kosong/error: Runtime > Change runtime type > T4 GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Step 0 — Install & import

In [ ]:
!pip install -q datasets scikit-learn seaborn
import os, re, json, math, shutil, random, hashlib
import numpy as np
import tensorflow as tf
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU tersedia:", tf.config.list_physical_devices("GPU"))

## Step 1 — Kelas kanonik (10 kelas tomat)

Urutan **harus identik** dengan v1 / `ml-service/models/label_map.json` (indeks 0–9), supaya `disease_reference` dan service tidak perlu berubah.

In [ ]:
TOMATO_CLASSES = [
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
]
assert TOMATO_CLASSES == sorted(TOMATO_CLASSES), "urutan harus alfabetis (sama dengan v1)"

def norm(s):
    return re.sub(r"[^a-z0-9]", "", s.lower())

CANON_BY_NORM = {norm(c): c for c in TOMATO_CLASSES}

DATA_DIR = "/content/data_v2"
SPLITS = ["lab_train", "lab_val", "lab_test", "field_train", "field_val", "field_test"]
shutil.rmtree(DATA_DIR, ignore_errors=True)
for sp in SPLITS:
    for c in TOMATO_CLASSES:
        os.makedirs(f"{DATA_DIR}/{sp}/{c}", exist_ok=True)

# registry: split -> list of (path, class_name, dhash)
records = {sp: [] for sp in SPLITS}

def dhash(img, size=8):
    g = img.convert("L").resize((size + 1, size), Image.LANCZOS)
    a = np.asarray(g, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)

def save_image(img, split, cls, idx):
    path = f"{DATA_DIR}/{split}/{cls}/{split}_{idx:06d}.jpg"
    img.convert("RGB").save(path, quality=95)
    return path

## Step 2 — PlantVillage (lab) → `lab_train` / `lab_val` / `lab_test`

Split asli dataset (per daun) dipakai untuk `lab_test`. `lab_val` diambil dari train **dengan `GroupShuffleSplit` per `leaf_id`**, lalu dicek tidak ada `leaf_id` yang bocor antar set.

In [ ]:
# mohanty/PlantVillage adalah dataset berbasis skrip (plant_village.py) yang TIDAK bisa dimuat dengan
# load_dataset() di versi `datasets` terbaru (BuilderConfig 'color' not found). Jadi file mentahnya dibaca
# langsung: data.zip (raw/color/<kelas>/<file>), splits/color_{train,test}.txt, leaf_grouping/leaf-map.json.
from huggingface_hub import hf_hub_download
import zipfile
from sklearn.model_selection import GroupShuffleSplit

REPO = "mohanty/PlantVillage"
zip_path = hf_hub_download(REPO, "data.zip", repo_type="dataset")   # +-2,2 GB
leaf_map = json.load(open(hf_hub_download(REPO, "leaf_grouping/leaf-map.json", repo_type="dataset"), encoding="utf-8"))
split_lines = {
    s: open(hf_hub_download(REPO, f"splits/color_{s}.txt", repo_type="dataset"), encoding="utf-8").read().splitlines()
    for s in ["train", "test"]
}
print("contoh baris split:", split_lines["train"][:2])

zf = zipfile.ZipFile(zip_path)
member_by_rel = {}
for n in zf.namelist():
    if "raw/color/Tomato" in n and not n.endswith("/"):
        member_by_rel[n[n.index("raw/color/"):]] = n
print("file tomat berwarna di zip:", len(member_by_rel), "| contoh:", list(member_by_rel)[:2])
assert member_by_rel, "tidak ada raw/color/Tomato* di data.zip — cek struktur: zf.namelist()[:10]"

def leaf_id_for(filename, class_name):
    # meniru turunan leaf_id di plant_village.py (hasil kira-kira; lihat pengecekan kebocoran di bawah)
    name = filename.replace("_final_masked", "")
    if "___" in name:
        name = name.split("___")[-1]
    low = name.lower()
    if "copy" in low:
        name = name[: low.index("copy")]
    for ext in (".jpg", ".png"):
        if name.lower().endswith(ext):
            name = name[: -len(ext)]
    ident = name.strip().lower()
    sugg = leaf_map.get(ident)
    if isinstance(sugg, str):
        sugg = [sugg]
    if sugg:
        if len(sugg) == 1:
            return str(sugg[0])
        for sg in sugg:
            if class_name in str(sg):
                return str(sg)
    return f"fallback_{ident}"

def parse(lines):
    out = []
    for ln in lines:
        parts = ln.strip().split("/")
        if len(parts) < 4 or parts[1] != "color":
            continue
        if norm(parts[2]) not in CANON_BY_NORM:
            continue
        rel = "/".join(parts)
        if rel in member_by_rel:
            out.append((rel, CANON_BY_NORM[norm(parts[2])], leaf_id_for(parts[-1], parts[2])))
    return out

pv_train_items, pv_test_items = parse(split_lines["train"]), parse(split_lines["test"])
print("PlantVillage tomat — train:", len(pv_train_items), "test:", len(pv_test_items))
assert pv_train_items and pv_test_items, "parse() menghasilkan 0 baris — bandingkan contoh baris split dengan format yang diharapkan (raw/color/<kelas>/<file>)"
n_fb = sum(1 for it in pv_train_items + pv_test_items if it[2].startswith("fallback_"))
print(f"leaf_id fallback (tidak ada di leaf-map): {n_fb} dari {len(pv_train_items) + len(pv_test_items)}")

In [ ]:
# Pengecekan kebocoran per daun: buang dari train semua gambar yang leaf_id-nya juga ada di test
test_leaves = {it[2] for it in pv_test_items}
before = len(pv_train_items)
pv_train_items = [it for it in pv_train_items if it[2] not in test_leaves]
print("gambar train dibuang karena leaf_id ada di test:", before - len(pv_train_items))

# lab_val dipisah per leaf_id dari train
leaf_train = np.array([it[2] for it in pv_train_items])
gss = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=SEED)
tr_idx, va_idx = next(gss.split(np.zeros(len(leaf_train)), groups=leaf_train))
assert not (set(leaf_train[tr_idx]) & set(leaf_train[va_idx])), "ada leaf_id yang sama di train dan val"
assert not (set(leaf_train) & test_leaves), "ada leaf_id yang sama di train dan test"
print("OK — tidak ada leaf_id yang sama antara lab_train, lab_val, dan lab_test")

def dump(items, indices, split):
    for i, k in enumerate(indices):
        rel, cls, _ = items[int(k)]
        with zf.open(member_by_rel[rel]) as fh:
            img = Image.open(fh).convert("RGB")
        path = save_image(img, split, cls, i)
        records[split].append((path, cls, dhash(img)))

dump(pv_train_items, tr_idx, "lab_train")
dump(pv_train_items, va_idx, "lab_val")
dump(pv_test_items, range(len(pv_test_items)), "lab_test")
print({sp: len(records[sp]) for sp in ["lab_train", "lab_val", "lab_test"]})

## Step 3 — PlantDoc (non-lab) → `field_train` / `field_val` / `field_test`

Pemetaan kelas PlantDoc → label kanonik. Yang tidak bisa dipetakan **tidak dipakai**. Duplikat (hash gambar sama) dibuang sebelum split; gambar yang muncul di dua kelas berbeda dibuang seluruhnya.

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

pdoc = load_dataset("Project-AgML/plant_doc_classification", split="train")
pdoc_names = pdoc.features["label"].names
print("Kelas PlantDoc:", len(pdoc_names))

PLANTDOC_TO_CANON = {
    "Tomato Early blight leaf": "Tomato___Early_blight",
    "Tomato Septoria leaf spot": "Tomato___Septoria_leaf_spot",
    "Tomato leaf": "Tomato___healthy",
    "Tomato leaf bacterial spot": "Tomato___Bacterial_spot",
    "Tomato leaf late blight": "Tomato___Late_blight",
    "Tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus",
    "Tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato mold leaf": "Tomato___Leaf_Mold",
    # "Tomato two spotted spider mites leaf": hanya 2 gambar -> dikecualikan (terlalu sedikit untuk di-split)
    # Target Spot: tidak ada di PlantDoc
}
missing = [k for k in PLANTDOC_TO_CANON if k not in pdoc_names]
assert not missing, f"nama kelas PlantDoc tidak ditemukan (nama berubah?): {missing}. Daftar: {pdoc_names}"
pdoc_idx_to_canon = {pdoc_names.index(k): v for k, v in PLANTDOC_TO_CANON.items()}

# kumpulkan gambar tomat PlantDoc (di memori; jumlahnya kecil, <1000)
pd_items = []
for ex in pdoc.filter(lambda lab: lab in pdoc_idx_to_canon, input_columns=["label"]):
    img = ex["image"].convert("RGB")
    pd_items.append((img, pdoc_idx_to_canon[ex["label"]], dhash(img)))
print("PlantDoc tomat (dipetakan):", len(pd_items))

# buang duplikat
by_hash = {}
for k, (_, cls, h) in enumerate(pd_items):
    by_hash.setdefault(h, []).append(k)
keep = []
dropped_dup = dropped_conflict = 0
for h, ks in by_hash.items():
    classes = {pd_items[k][1] for k in ks}
    if len(classes) > 1:
        dropped_conflict += len(ks)
    else:
        keep.append(ks[0]); dropped_dup += len(ks) - 1
print(f"duplikat dibuang: {dropped_dup}; gambar berlabel ganda dibuang: {dropped_conflict}; tersisa: {len(keep)}")

labels_keep = [pd_items[k][1] for k in keep]
# kelas dengan <4 gambar tidak bisa di-stratify 3 arah -> dikecualikan dengan peringatan
from collections import Counter
cnt = Counter(labels_keep)
too_small = {c for c, n in cnt.items() if n < 4}
if too_small:
    print("PERINGATAN kelas dikecualikan (terlalu sedikit):", too_small)
keep = [k for k in keep if pd_items[k][1] not in too_small]
labels_keep = [pd_items[k][1] for k in keep]

k_trval, k_test = train_test_split(keep, test_size=0.30, stratify=labels_keep, random_state=SEED)
k_train, k_val = train_test_split(k_trval, test_size=0.26, stratify=[pd_items[k][1] for k in k_trval], random_state=SEED)

for split, ks in [("field_train", k_train), ("field_val", k_val), ("field_test", k_test)]:
    for i, k in enumerate(ks):
        img, cls, h = pd_items[k]
        path = save_image(img, split, cls, i)
        records[split].append((path, cls, h))
print({sp: len(records[sp]) for sp in ["field_train", "field_val", "field_test"]})

## Step 4 — Anti-kebocoran: buang near-duplicate lintas set

Hash perseptual (dHash 64-bit). Gambar di set yang lebih "bersih" menjadi acuan: `field_test` dan `field_val` tidak boleh punya near-duplicate (jarak Hamming ≤ 3) di set latih.

In [ ]:
def hashes_of(split):
    return np.array([h for _, _, h in records[split]], dtype=np.uint64)

def popcount64(a):
    return np.unpackbits(np.ascontiguousarray(a).view(np.uint8).reshape(-1, 8), axis=1).sum(axis=1)

def purge(from_splits, against_split, thr=3):
    ref = hashes_of(against_split)
    removed = 0
    for sp in from_splits:
        kept = []
        for rec in records[sp]:
            h = np.uint64(rec[2])
            if len(ref) and (popcount64(ref ^ h).min() <= thr):
                os.remove(rec[0]); removed += 1
            else:
                kept.append(rec)
        records[sp] = kept
    return removed

r1 = purge(["lab_train", "lab_val", "lab_test", "field_train", "field_val"], "field_test")
r2 = purge(["lab_train", "field_train"], "field_val")
print("near-duplicate dibuang terhadap field_test:", r1, "| terhadap field_val:", r2)

## Step 5 — Audit dataset (jumlah per kelas per set)

In [ ]:
import pandas as pd

rows = []
for c in TOMATO_CLASSES:
    rows.append({"kelas": c.replace("Tomato___", "")} | {sp: sum(1 for r in records[sp] if r[1] == c) for sp in SPLITS})
audit = pd.DataFrame(rows).set_index("kelas")
audit.loc["TOTAL"] = audit.sum()
display(audit)
print("Kelas tanpa data lapangan (uji lapangan tidak mencakupnya):",
      [c for c in TOMATO_CLASSES if not any(r[1] == c for r in records["field_test"])])

## Step 6 — Modul pra-pemrosesan (diambil dari repo, identik dengan yang dipakai API)

In [ ]:
!rm -rf /content/agrivision && git clone -q https://github.com/Argy1/AgriVision.git /content/agrivision
import sys
sys.path.insert(0, "/content/agrivision/ml-service")
from app.ml.enhance import apply_variant, VARIANTS
print("Varian pra-pemrosesan:", list(VARIANTS))

## Step 7 — Bangun dataset per varian

Setiap gambar: buka → resize 224×224 (PIL bilinear, seperti di API) → `apply_variant` → simpan PNG (lossless). Dataset varian dibuat sekali lalu dipakai ulang untuk semua seed.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

IMG_SIZE = (224, 224)
VAR_ROOT = "/content/data_v3"

def _convert(args):
    src, dst, variant = args
    img = Image.open(src).convert("RGB").resize(IMG_SIZE, Image.BILINEAR)
    Image.fromarray(apply_variant(np.asarray(img, dtype=np.uint8), variant)).save(dst)

def build_variant(variant):
    root = f"{VAR_ROOT}/{variant}"
    if os.path.isdir(root):
        return root
    jobs = []
    for sp in SPLITS:
        for c in TOMATO_CLASSES:
            os.makedirs(f"{root}/{sp}/{c}", exist_ok=True)
        for path, cls, _h in records[sp]:
            jobs.append((path, f"{root}/{sp}/{cls}/{os.path.basename(path)[:-4]}.png", variant))
    with ThreadPoolExecutor(8) as ex:
        list(ex.map(_convert, jobs))
    return root

import time
t0 = time.time()
build_variant("none")
print(f"dataset 'none' siap ({time.time() - t0:.0f} dtk)")

## Step 8 — Pipeline data, model MobileNetV2, dan prosedur latih

Augmentasi (di GPU, di dalam model) dan semua hiperparameter identik untuk setiap varian. Tahap 1: kepala klasifikasi (Adam 1e-3, ≤8 epoch). Tahap 2: 60 lapisan atas dibuka kecuali BatchNorm (Adam 1e-4, ≤15 epoch). Early stopping memantau akurasi pada gabungan `lab_val` + `field_val`.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

BATCH = 32
FIELD_WEIGHT = 0.25
AUTOTUNE = tf.data.AUTOTUNE

def make_augment():
    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomRotation(0.1),
        tf.keras.layers.RandomZoom(0.15),
        tf.keras.layers.RandomTranslation(0.05, 0.05),
        tf.keras.layers.RandomBrightness(0.15, value_range=(0, 255)),
        tf.keras.layers.RandomContrast(0.15),
    ], name="augment")

def load_dir_v(root, split, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        f"{root}/{split}", class_names=TOMATO_CLASSES, label_mode="int",
        image_size=IMG_SIZE, batch_size=None, shuffle=shuffle, seed=SEED,
    )

def to_u8(x, y):
    return tf.cast(x, tf.uint8), y

def to_f32(x, y):
    return tf.cast(x, tf.float32), y

def make_train_ds(root):
    lab = load_dir_v(root, "lab_train", True).map(to_u8).cache().shuffle(2000, seed=SEED).repeat()
    fld = load_dir_v(root, "field_train", True).map(to_u8).cache().shuffle(1000, seed=SEED).repeat()
    mixed = tf.data.Dataset.sample_from_datasets([lab, fld], weights=[1 - FIELD_WEIGHT, FIELD_WEIGHT], seed=SEED)
    return mixed.batch(BATCH).map(to_f32, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

def make_eval_ds(root, split):
    return load_dir_v(root, split, False).map(to_u8).cache().map(to_f32).batch(BATCH).prefetch(AUTOTUNE)

def make_val_ds(root):
    ds = load_dir_v(root, "lab_val", False).concatenate(load_dir_v(root, "field_val", False))
    return ds.map(to_u8).cache().map(to_f32).batch(BATCH).prefetch(AUTOTUNE)

n_lab, n_fld = len(records["lab_train"]), len(records["field_train"])
STEPS_PER_EPOCH = (n_lab + n_fld) // BATCH
counts = np.array([sum(1 for r in records["lab_train"] + records["field_train"] if r[1] == c) for c in TOMATO_CLASSES], dtype=float)
CLASS_WEIGHT = {i: float(counts.sum() / (len(counts) * max(n, 1))) for i, n in enumerate(counts)}

def build_mobilenet():
    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)  # input mentah 0-255 -> [-1, 1] (di dalam graf)
    base = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
    base.trainable = False
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(len(TOMATO_CLASSES), activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name="mobilenetv2_tomato"), base

def n_trainable(m):
    return int(sum(np.prod(w.shape) for w in m.trainable_weights))

def compile_model(model, lr):
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

def train_run(variant, seed):
    root = build_variant(variant)
    tf.keras.utils.set_random_seed(seed)
    core, base = build_mobilenet()
    model = tf.keras.Sequential([tf.keras.Input(shape=IMG_SIZE + (3,)), make_augment(), core], name="train_wrapper")
    train_ds, val_ds = make_train_ds(root), make_val_ds(root)
    stop = lambda p: [tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=p, restore_best_weights=True)]
    t0 = time.time()

    compile_model(model, 1e-3)
    trainable_stage1 = n_trainable(core)
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=8, steps_per_epoch=STEPS_PER_EPOCH,
                   class_weight=CLASS_WEIGHT, callbacks=stop(3), verbose=0)

    base.trainable = True
    for layer in base.layers[:-60]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    compile_model(model, 1e-4)
    trainable_stage2 = n_trainable(core)
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=15, steps_per_epoch=STEPS_PER_EPOCH,
                   class_weight=CLASS_WEIGHT, callbacks=stop(4), verbose=0)
    info = {
        "epochs_stage1": len(h1.history["loss"]), "epochs_stage2": len(h2.history["loss"]),
        "best_val_acc": float(max(h1.history["val_accuracy"] + h2.history["val_accuracy"])),
        "train_acc_last": float(h2.history["accuracy"][-1]),
        "trainable_stage1": trainable_stage1, "trainable_stage2": trainable_stage2,
        "total_params": int(core.count_params()), "seconds": time.time() - t0,
    }
    return core, info, root

def predict_split(core, root, split):
    ds = make_eval_ds(root, split)
    y_true = np.concatenate([y.numpy() for _, y in ds])
    probs = core.predict(ds, verbose=0)
    return y_true, probs.argmax(1)

def run_metrics(y_true, y_pred):
    present = sorted(set(y_true.tolist()))
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=present, zero_division=0)
    _, _, fw, _ = precision_recall_fscore_support(y_true, y_pred, labels=present, average="weighted", zero_division=0)
    return {"n": int(len(y_true)), "accuracy": float((y_true == y_pred).mean()),
            "macro_precision": float(p.mean()), "macro_recall": float(r.mean()),
            "macro_f1": float(f.mean()), "weighted_f1": float(fw)}

RESULTS = {}   # (variant, seed) -> {"info", "metrics": {split: {...}}, "preds": {split: (y_true, y_pred)}}

def run_all(variants, seeds):
    for v in variants:
        for sd in seeds:
            if (v, sd) in RESULTS:
                continue
            core, info, root = train_run(v, sd)
            preds = {sp: predict_split(core, root, sp) for sp in ["lab_val", "field_val", "lab_test", "field_test"]}
            metrics = {sp: run_metrics(*preds[sp]) for sp in preds}
            RESULTS[(v, sd)] = {"info": info, "metrics": metrics, "preds": preds}
            m = metrics
            print(f"[{v:13s} seed={sd}] ep={info['epochs_stage1']}+{info['epochs_stage2']} "
                  f"lab_test={m['lab_test']['accuracy']*100:.2f}% field_test={m['field_test']['accuracy']*100:.2f}% "
                  f"field_val={m['field_val']['accuracy']*100:.2f}% ({info['seconds']/60:.1f} mnt)")
            del core
            tf.keras.backend.clear_session()

## Step 9 — Jalankan semua varian (seed 42)

In [ ]:
run_all(list(VARIANTS), [42])

## Step 10 — Tabel ringkas, pemilihan varian, dan seed tambahan

Varian dipilih berdasarkan akurasi `field_val` (bukan `field_test`). Baseline (`none`) dan dua varian terbaik dijalankan ulang dengan seed tambahan untuk melihat variansi antarseed.

In [ ]:
import pandas as pd

def summary_table():
    rows = []
    for v in VARIANTS:
        runs = [RESULTS[k] for k in RESULTS if k[0] == v]
        if not runs:
            continue
        def stat(split, key):
            vals = np.array([r["metrics"][split][key] for r in runs]) * 100
            return f"{vals.mean():.2f} ± {vals.std():.2f}" if len(vals) > 1 else f"{vals.mean():.2f}"
        rows.append({"varian": v, "seed": len(runs),
                     "lab_test acc": stat("lab_test", "accuracy"), "lab_test F1": stat("lab_test", "macro_f1"),
                     "field_test acc": stat("field_test", "accuracy"), "field_test F1": stat("field_test", "macro_f1"),
                     "field_val acc": stat("field_val", "accuracy")})
    return pd.DataFrame(rows).set_index("varian")

display(summary_table())
mean_field_val = {v: np.mean([RESULTS[k]["metrics"]["field_val"]["accuracy"] for k in RESULTS if k[0] == v]) for v in VARIANTS}
TOP = [v for v, _ in sorted(((v, a) for v, a in mean_field_val.items() if v != "none"), key=lambda t: -t[1])[:2]]
print("Dua varian terbaik (field_val):", TOP)
run_all(["none"] + TOP, [1, 2])
display(summary_table())
mean_field_val = {v: np.mean([RESULTS[k]["metrics"]["field_val"]["accuracy"] for k in RESULTS if k[0] == v]) for v in VARIANTS if any(k[0] == v for k in RESULTS)}
BEST = max((v for v in mean_field_val if v != "none"), key=lambda v: mean_field_val[v])
print("Varian terbaik (rata-rata field_val, tanpa baseline):", BEST)

## Step 11 — Uji signifikansi terhadap baseline (seed 42, prediksi berpasangan)

In [ ]:
from scipy.stats import binomtest

def mcnemar(y, pa, pb):
    a_ok, b_ok = pa == y, pb == y
    n10 = int((a_ok & ~b_ok).sum())   # baseline benar, varian salah
    n01 = int((~a_ok & b_ok).sum())   # baseline salah, varian benar
    p = binomtest(min(n10, n01), n10 + n01, 0.5).pvalue if n10 + n01 else 1.0
    return n10, n01, p

def boot_diff(y, pa, pb, B=2000):
    rng = np.random.default_rng(0)
    diffs = []
    for _ in range(B):
        idx = rng.integers(0, len(y), len(y))
        diffs.append((pb[idx] == y[idx]).mean() - (pa[idx] == y[idx]).mean())
    return np.percentile(diffs, [2.5, 97.5]) * 100

rows = []
for v in VARIANTS:
    if v == "none" or ("none", 42) not in RESULTS or (v, 42) not in RESULTS:
        continue
    for split in ["lab_test", "field_test"]:
        y, pa = RESULTS[("none", 42)]["preds"][split]
        _, pb = RESULTS[(v, 42)]["preds"][split]
        n10, n01, p = mcnemar(y, pa, pb)
        lo, hi = boot_diff(y, pa, pb)
        rows.append({"varian": v, "set": split, "selisih akurasi (poin)": f"{((pb == y).mean() - (pa == y).mean()) * 100:+.2f}",
                     "CI95 bootstrap": f"[{lo:+.1f}, {hi:+.1f}]", "base benar/varian salah": n10,
                     "base salah/varian benar": n01, "p McNemar": f"{p:.3f}"})
display(pd.DataFrame(rows))
print("Catatan: p < 0,05 pada banyak varian sekaligus perlu dikoreksi (Bonferroni: bagi 0,05 dengan jumlah varian).")

## Step 12 — Laporan lengkap baseline dan varian terbaik: P/R/F1 per kelas, matriks kebingungan, model summary, tabel dataset

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

OUT3 = "/content/outputs_v3"
os.makedirs(OUT3, exist_ok=True)
short = [c.replace("Tomato___", "") for c in TOMATO_CLASSES]

for name in ["none", BEST]:
    for split in ["lab_test", "field_test"]:
        y, p = RESULTS[(name, 42)]["preds"][split]
        present = sorted(set(y.tolist()))
        print(f"\n=== {name} — {split} (n={len(y)}) ===")
        print(classification_report(y, p, labels=present, target_names=[short[i] for i in present], digits=4, zero_division=0))
        cm = confusion_matrix(y, p, labels=list(range(len(short))))
        plt.figure(figsize=(9, 7))
        sns.heatmap(cm, annot=True, fmt="d", xticklabels=short, yticklabels=short, cmap="Greens")
        plt.title(f"Confusion Matrix — {name} — {split}"); plt.ylabel("Aktual"); plt.xlabel("Prediksi")
        plt.xticks(rotation=45, ha="right"); plt.tight_layout()
        plt.savefig(f"{OUT3}/confusion_{name}_{split}.png", dpi=150); plt.show()

# Model summary MobileNetV2 (arsitektur sama untuk semua varian)
core, base = build_mobilenet()
lines = []
core.summary(print_fn=lambda s, **kw: lines.append(s))
open(f"{OUT3}/model_summary.txt", "w", encoding="utf-8").write("\n".join(lines))
print("\n".join(lines[-12:]))
info = RESULTS[("none", 42)]["info"]
print(f"\nParameter total: {info['total_params']:,} | dilatih tahap 1 (kepala): {info['trainable_stage1']:,} | dilatih tahap 2 (60 lapisan atas + kepala, BN dibekukan): {info['trainable_stage2']:,}")

audit.to_csv(f"{OUT3}/dataset_table.csv")
display(audit)

In [ ]:
summary_table().to_csv(f"{OUT3}/ringkasan_varian.csv")
out = {f"{k[0]}|seed{k[1]}": {"info": v["info"], "metrics": v["metrics"]} for k, v in RESULTS.items()}
json.dump(out, open(f"{OUT3}/hasil_lengkap.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False, default=float)
print("Tersimpan di", OUT3, "->", sorted(os.listdir(OUT3)))

## Setelah selesai
Kembali ke sesi Claude Code proyek ini dan sampaikan hasil Step 10–12. Varian terbaik dipasang ke API hanya bila (a) lebih baik pada `field_test` dibanding baseline dengan bukti yang masuk akal dan (b) tidak merusak `lab_test`; kalau tidak, hasil dilaporkan apa adanya (hasil negatif tetap temuan).